In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import statsmodels.formula.api as smf
import pickle

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, roc_curve, auc, recall_score, precision_recall_curve
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, precision_score, hamming_loss
from sklearn.metrics import brier_score_loss, average_precision_score

import copy 
import torch
import torch.nn as nn

In [2]:

import pickle
with open('balanced_data.pkl', 'rb') as b:
    data = pickle.load(b)

X_res_sk = data['X_res_stroke']
y_res_sk = data['y_res_stroke']
X_res_db = data['X_res_diab']
y_res_db = data['y_res_diab']
X_res_cn = data['X_res_can']
y_res_cn = data['y_res_can']       
X_res_ha = data['X_res_ha']
y_res_ha = data['y_res_ha']    

X_val = data['X_val'] 
y_val = data['y_val']   

X_test = data['X_test']
y_test = data['y_test']

X_bal=data['X_bal']
y_bal=data['y_bal']
    

In [3]:
y_test=y_test.rename(columns=
              {'heart_attack' : 'HA'})
y_bal=y_bal.rename(columns=
              {'heart_attack' : 'HA'})
y_val=y_val.rename(columns=
              {'heart_attack' : 'HA'})

In [ ]:
scaler = MinMaxScaler()
X_train_st = scaler.fit_transform(X_bal)    
X_val_st   = scaler.transform(X_val)        
X_test_st = scaler.transform(X_test)      

x_train_t = torch.tensor(X_train_st, dtype=torch.float32)
y_train_t = torch.tensor(y_bal.values, dtype=torch.float32)

x_val_t   = torch.tensor(X_val_st, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

x_test_t  = torch.tensor(X_test_st, dtype=torch.float32)

In [5]:
def youden_threshold(y_true_col, y_prob_col):
    fpr, tpr, thresholds = roc_curve(y_true_col, y_prob_col)
    return thresholds[np.argmax(tpr - fpr)]

***MLP DROPOUT MONTECARLO***

In [ ]:

class MLP_MC(nn.Module):
    def __init__(self, n_features, n_outputs, M, drop_rate = 0.2): 
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(n_features, M),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(M, M),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(M, n_outputs)
        )
        
    def forward(self, x):
        return self.model(x)
    
    def drop_enable(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

In [ ]:
torch.manual_seed(1234)
n_features = x_train_t.shape[1]
n_outputs = y_bal.shape[1]

pos_weight = torch.tensor(
    [
        ((y_bal[col] == 0).sum()) / ((y_bal[col] == 1).sum())
        for col in y_bal.columns
    ], dtype= torch.float32
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight = pos_weight)

n_epochs = 1000
n_restarts = 5
n_sample = 30

lambdas = np.logspace(-5,-3,4)
M_list = [16, 32, 64]
drop_rates = [0.05, 0.1, 0.15]

tuning_result = []

for lam in lambdas:
    for M in M_list:
        for rate in drop_rates:
            
            best_misc = np.inf
            best_model = None
            
            for h in range(n_restarts):
                model = MLP_MC(n_features, n_outputs, M, drop_rate = rate)
                opt = torch.optim.Adam(model.parameters(), lr= 0.001, weight_decay=lam)
                
                for epoch in range(n_epochs):
                    model.train()
                    opt.zero_grad()
                    loss = loss_fn(model(x_train_t),y_train_t)
                    loss.backward()
                    opt.step()
                    
                
                model.eval()
                model.drop_enable()
                with torch.no_grad():
                    y_prob_val_h = torch.stack([
                        torch.sigmoid(model(x_val_t))
                        for _ in range(n_sample)]).mean(0).numpy()
                
                errors_h = []
                for i, col in enumerate(y_val.columns):
                    thr = youden_threshold(y_val[col].values, y_prob_val_h[:,i])
                    y_pred_i = (y_prob_val_h[:,i] >= thr).astype(int)
                    errors_h.append((y_pred_i != y_val[col].values).mean())
                misc_h = np.mean(errors_h)
                
                if misc_h < best_misc:
                    best_misc = misc_h
                    best_model = copy.deepcopy(model)
            
            best_model.eval()
            best_model.drop_enable()
            with torch.no_grad():
                y_prob_val = torch.stack(
                    [torch.sigmoid(best_model(x_val_t))
                     for _ in range(n_sample)]
                ).mean(0).numpy()
            
            error = []
            for i, col in enumerate(y_val.columns):
                thr = youden_threshold(y_val[col].values, y_prob_val[:,i])
                y_pred_i = (y_prob_val[:,i] >= thr).astype(int)
                error.append(
                    (y_pred_i != y_val[col].values).mean()
                )
            misc = np.mean(error)
            
            print(f'lambda = {lam:.6f}, M = {M}, droprate = {rate}, misc = {misc:.4f}')
            tuning_result.append((lam, M, rate, misc))

best_lam, best_M, best_rate, best_misclass = min(tuning_result, key= lambda x: x[3])
print(f'best lambda = {best_lam:.6f}, best M = {best_M}, best drop rate = {best_rate}, misc = {best_misclass:.4f}')

In [8]:
torch.manual_seed(1234)
model_final = None
misc_final = np.inf

for h in range(n_restarts):
    model = MLP_MC(n_features, n_outputs, best_M, drop_rate= best_rate)
    opt = torch.optim.Adam(model.parameters(), lr = 0.001, weight_decay=best_lam)
    
    for epoch in range(n_epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(x_train_t), y_train_t)
        loss.backward()
        opt.step()
        
    model.eval()
    model.drop_enable()
    with torch.no_grad():
        y_prob_val_h = torch.stack([
            torch.sigmoid(model(x_val_t))
            for _ in range(n_sample)]
        ).mean(0).numpy()
            
    error_h = []
    for i, col in enumerate(y_val.columns):
        thr = youden_threshold(y_val[col].values, y_prob_val_h[:,i])
        y_pred_i = (y_prob_val_h[:,i] >= thr).astype(int)
        error_h.append((y_pred_i != y_val[col].values).mean())
    misc_h = np.mean(error_h)
        
    if misc_h < misc_final:
        misc_final = misc_h
        model_final = copy.deepcopy(model)
        
print(f'best misclassification error = {misc_final:.4f}')

best misclassification error = 0.3169


In [ ]:

model_final.eval()
model_final.drop_enable()
n_sample_test = 100

with torch.no_grad():
    pred = torch.stack([
        torch.sigmoid(model_final(x_test_t))
        for _ in range(n_sample_test)
    ])

y_prob_test = pred.mean(0).numpy()
y_std_test = pred.std(0).numpy()


best_thr = {}
model_final.eval()
model_final.drop_enable()
with torch.no_grad():
    pred_val = torch.stack([
        torch.sigmoid(model_final(x_val_t))
        for _ in range(n_sample_test)
    ])

y_prob_val = pred_val.mean(0).numpy()

for i, col in enumerate(y_val.columns):
    thr = youden_threshold(y_val[col].values, y_prob_val[:,i])
    best_thr[col] = thr
    print(f'{col}, thr = {thr:.4f}')


results_mlp_mc = {}
for i, col in enumerate(y_test.columns):
    y_pred_i = (y_prob_test[:,i] >= best_thr[col]).astype(int)
    precision, recall, _ = precision_recall_curve(y_test[col].values, y_prob_test[:,i])
    
    p = y_prob_test[:,i]
    eps = 1e-8
    entropy_i = -p * np.log2(p+eps) - (1-p)* np.log2(1-p+eps)
    
    results_mlp_mc[col] = {
        'f1' : f1_score(y_test[col], y_pred_i),
        'precision' : precision_score(y_test[col],y_pred_i),
        'auc' : roc_auc_score(y_test[col], y_prob_test[:,i]),
        'auc-pr' : auc(recall, precision),
        'sens' : recall_score(y_test[col].values, y_pred_i),
        'uncertainty' : y_std_test[:,i].mean()*100,
        'ambiguity' : entropy_i.mean(),
        'confusion_matrix' : confusion_matrix(y_test[col], y_pred_i),
        'misc' : (y_pred_i != y_test[col].values).mean(),
        "brier": brier_score_loss(y_test[col].values, y_prob_test[:, i]),
        'threshold' : best_thr[col]
    }

y_pred_multi = np.zeros_like(y_test.values)
for i, col in enumerate(y_test.columns):
    y_pred_multi[:, i] = (y_prob_test[:, i] >= best_thr[col]).astype(int)

macro_f1 = f1_score(y_test.values, y_pred_multi, average='macro')

hamming = hamming_loss(y_test.values, y_pred_multi)

brier_mean = np.mean([
    brier_score_loss(y_test[col].values, y_prob_test[:, i])
    for i, col in enumerate(y_test.columns)
])

macro_auc = np.mean([
    roc_auc_score(y_test[col].values, y_prob_test[:, i])
    for i, col in enumerate(y_test.columns)
])
macro_auc_pr = np.mean([
    average_precision_score(y_test[col].values, y_prob_test[:, i])
    for i, col in enumerate(y_test.columns)
])

results_mlp_mc["MacroAUC"] = macro_auc
results_mlp_mc["MacroAUC_PR"] = macro_auc_pr
results_mlp_mc["Brier_mean"] = brier_mean
results_mlp_mc["MacroF1"] = macro_f1
results_mlp_mc["Hamming"] = hamming

print(pd.DataFrame({k:v for k,v in results_mlp_mc.items()
                   if k not in ['confusion_matrix', 'threshold']}).T)

for col, res in results_mlp_mc.items():
    if isinstance(res, dict):
        print(f'\n{col}:\n', res['confusion_matrix'])

diabetes, thr = 0.5536
stroke, thr = 0.3793
HA, thr = 0.4701
cancer, thr = 0.2524
                   f1 precision       auc    auc-pr      sens uncertainty  \
diabetes     0.460076  0.334716  0.825424  0.437162  0.735562    5.873765   
stroke       0.116056  0.063541  0.733204  0.083729  0.668831    5.321679   
HA           0.329696  0.209613  0.750738   0.27314  0.771898    4.543139   
cancer       0.092643  0.049259   0.62041  0.067698   0.77665    6.133661   
MacroAUC     0.732444  0.732444  0.732444  0.732444  0.732444    0.732444   
MacroAUC_PR  0.216837  0.216837  0.216837  0.216837  0.216837    0.216837   
Brier_mean   0.166097  0.166097  0.166097  0.166097  0.166097    0.166097   
MacroF1      0.249618  0.249618  0.249618  0.249618  0.249618    0.249618   
Hamming      0.402407  0.402407  0.402407  0.402407  0.402407    0.402407   

            ambiguity            confusion_matrix      misc     brier  \
diabetes     0.807127   [[2991, 962], [174, 484]]  0.246367  0.182033   
s

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    quantiles = np.percentile(y_prob, np.linspace(0, 100, n_bins + 1))
    ece = 0
    n = len(y_prob)
    for b in range(n_bins):
        mask = (y_prob >= quantiles[b]) & (y_prob < quantiles[b+1])
        if mask.sum() > 0:
            acc = y_true[mask].mean()   
            conf = y_prob[mask].mean()  
            ece += mask.mean() * abs(acc - conf)
    return ece

for i, col in enumerate(y_test.columns):
    ece = expected_calibration_error(
        y_test[col].values, 
        y_prob_test[:, i]
    )
    print(f'{col}: ECE = {ece:.4f}')
    results_mlp_mc[col]["ece"] = ece


diabetes: ECE = 0.2688
stroke: ECE = 0.2854
HA: ECE = 0.3026
cancer: ECE = 0.2959


In [11]:
for i, col in enumerate(y_test.columns):
    p = y_prob_test[:, i]
    eps = 1e-8
    entropy_i = - p * np.log2(p + eps) - (1 - p) * np.log(1 - p + eps)
    
    mask_pos = y_test[col].values == 1
    mask_neg = y_test[col].values == 0
    
    print(f"\n{col}:")
    print(f"  ambiguity positivi : {entropy_i[mask_pos].mean():.4f}")
    print(f"  ambiguity negativi : {entropy_i[mask_neg].mean():.4f}")
    print(f"  mean_prob positivi : {p[mask_pos].mean():.4f}")
    print(f"  mean_prob negativi : {p[mask_neg].mean():.4f}")


diabetes:
  ambiguity positivi : 0.6996
  ambiguity negativi : 0.6892
  mean_prob positivi : 0.6416
  mean_prob negativi : 0.3734

stroke:
  ambiguity positivi : 0.7679
  ambiguity negativi : 0.6826
  mean_prob positivi : 0.4695
  mean_prob negativi : 0.3138

HA:
  ambiguity positivi : 0.7604
  ambiguity negativi : 0.7247
  mean_prob positivi : 0.5749
  mean_prob negativi : 0.4007

cancer:
  ambiguity positivi : 0.7692
  ambiguity negativi : 0.7383
  mean_prob positivi : 0.4005
  mean_prob negativi : 0.3360


In [ ]:

with torch.no_grad():
    prova = torch.stack([
        model_final(x_test_t)
        for _ in range(n_sample)]) 

prova_std = prova.std(0).numpy()
print(prova_std)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_final = model_final.to(device)

def predict_single_patient(model, x_row, thresholds, n_samples=100):
    """
    - prob_mean: sigmoid
    - prob_std: std MC, uncertainty
    - prediction: classes 0/1 considering Youden Index
    """
    x_np = np.array(x_row).reshape(1, -1)
    x_t  = torch.tensor(x_np, dtype=torch.float32).to(device)

    model.eval()
    model.drop_enable()
    samples = []

    with torch.no_grad():
        for _ in range(n_samples):
            out = model(x_t)
            prob = torch.sigmoid(out).cpu().numpy()
            samples.append(prob)

    samples = np.vstack(samples)   

    prob_mean = samples.mean(axis=0)
    prob_std  = samples.std(axis=0)
    
    eps = 1e-8
    ambiguity = - prob_mean * np.log2(prob_mean + eps) \
                - (1 - prob_mean) * np.log2(1 - prob_mean + eps)

    thr_vec = np.array([thresholds[c] for c in thresholds])
    prediction = (prob_mean >= thr_vec).astype(int)

    return prob_mean, prob_std, prediction, ambiguity

[[0.29445603 0.28216937 0.2145569  0.2885078 ]
 [0.4074795  0.33449182 0.30164674 0.34863585]
 [0.33482802 0.2924334  0.20645954 0.21660832]
 ...
 [0.28571615 0.3194654  0.30682805 0.3337748 ]
 [0.37457135 0.2478704  0.29605302 0.2609853 ]
 [0.16131611 0.3346665  0.15546298 0.36345875]]


In [13]:
patient_idx=1810
x_one = X_test.iloc[patient_idx].values.reshape(1, -1)
x_one_std = scaler.transform(x_one)  

prob, unc, pred, ambiguity = predict_single_patient(
    model_final,
    x_one_std[0],
    best_thr,
    n_samples=100
)

df_patient = pd.DataFrame({
    "prob": prob,
    "uncertainty": unc *100,  # %
    "ambiguity" : ambiguity,
    "threshold": [best_thr[c] for c in y_test.columns],
    "prediction": pred,
    "True Pred": [y_test.iloc[patient_idx][c] for c in y_test.columns]
}, index=y_test.columns)

print(df_patient)

              prob  uncertainty  ambiguity  threshold  prediction  True Pred
diabetes  0.089364     4.324445   0.434342   0.553643           0          0
stroke    0.283867     7.161000   0.860667   0.379285           0          0
HA        0.313114     5.830079   0.896736   0.470119           0          0
cancer    0.514074    10.792661   0.999428   0.252395           1          0


/Users/samuelemazzei/Desktop/Python_tesi/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [14]:
from pathlib import Path
import pickle

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "Models" else Path.cwd()

with open(PROJECT_DIR / "results_mc_dropout.pkl", "wb") as f:
    pickle.dump(results_mlp_mc, f)

print(f"Saved summary results to {PROJECT_DIR / 'results_mc_dropout.pkl'}")


Saved summary results to /Users/samuelemazzei/Desktop/Python_tesi/results_mc_dropout.pkl


In [15]:
from pathlib import Path
import pickle

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "Models" else Path.cwd()
uncertainty_rows = []
eps = 1e-8

for i, col in enumerate(y_test.columns):
    p = y_prob_test[:, i]
    uncertainty = y_std_test[:, i]
    ambiguity = -p * np.log2(p + eps) - (1 - p) * np.log2(1 - p + eps)
    threshold = best_thr[col]
    prediction = (p >= threshold).astype(int)
    true_label = y_test[col].values.astype(int)

    df_tmp = pd.DataFrame({
        "model": "MC Dropout",
        "patient_id": np.arange(len(p)),
        "label": col,
        "probability": p,
        "uncertainty": uncertainty,
        "uncertainty_pct": uncertainty * 100,
        "ambiguity": ambiguity,
        "threshold": threshold,
        "true_label": true_label,
        "prediction": prediction,
        "correct": prediction == true_label,
        "distance_from_threshold": np.abs(p - threshold),
    })
    df_tmp["error_type"] = np.select(
        [
            (df_tmp["true_label"] == 1) & (df_tmp["prediction"] == 1),
            (df_tmp["true_label"] == 0) & (df_tmp["prediction"] == 0),
            (df_tmp["true_label"] == 0) & (df_tmp["prediction"] == 1),
            (df_tmp["true_label"] == 1) & (df_tmp["prediction"] == 0),
        ],
        ["TP", "TN", "FP", "FN"],
        default="NA",
    )
    uncertainty_rows.append(df_tmp)

df_uncertainty = pd.concat(uncertainty_rows, ignore_index=True)

with open(PROJECT_DIR / "patient_level_mc_dropout.pkl", "wb") as f:
    pickle.dump(df_uncertainty, f)

df_uncertainty.to_csv(PROJECT_DIR / "patient_level_mc_dropout.csv", index=False)
print(f"Saved patient-level results to {PROJECT_DIR / 'patient_level_mc_dropout.pkl'}")
df_uncertainty.head()


Saved patient-level results to /Users/samuelemazzei/Desktop/Python_tesi/patient_level_mc_dropout.pkl


,model,patient_id,label,probability,uncertainty,uncertainty_pct,ambiguity,threshold,true_label,prediction,correct,distance_from_threshold,error_type
0,MC Dropout,0,diabetes,0.269488,0.056895,5.689539,0.840729,0.553643,0,0,True,0.284156,TN
1,MC Dropout,1,diabetes,0.051688,0.035899,3.589855,0.293523,0.553643,0,0,True,0.501956,TN
2,MC Dropout,2,diabetes,0.509325,0.057006,5.700618,0.999749,0.553643,0,0,True,0.044318,TN
3,MC Dropout,3,diabetes,0.206224,0.069665,6.966480,0.734203,0.553643,0,0,True,0.347419,TN
4,MC Dropout,4,diabetes,0.193698,0.059687,5.968707,0.709143,0.553643,0,0,True,0.359945,TN


In [ ]:
import seaborn as sns

def get_single_patient_mc_samples(model, x_row, n_samples=500):
    x_np = np.array(x_row).reshape(1, -1)
    x_t = torch.tensor(x_np, dtype=torch.float32).to(device)

    model.eval()
    model.drop_enable()
    samples = []

    with torch.no_grad():
        for _ in range(n_samples):
            out = model(x_t)
            prob = torch.sigmoid(out).cpu().numpy().ravel()
            samples.append(prob)

    return np.vstack(samples)


x_one = X_test.iloc[patient_idx].values.reshape(1, -1)
x_one_std = scaler.transform(x_one)

patient_samples = get_single_patient_mc_samples(
    model_final,
    x_one_std[0],
    n_samples=500
)

patient_mean = patient_samples.mean(axis=0)
patient_std = patient_samples.std(axis=0)
patient_thresholds = np.array([best_thr[col] for col in y_test.columns])
patient_pred = (patient_mean >= patient_thresholds).astype(int)
patient_true = y_test.iloc[patient_idx].values

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()

palette = sns.color_palette("Set2", n_colors=len(y_test.columns))

for i, (col, ax) in enumerate(zip(y_test.columns, axes)):
    sns.histplot(
        patient_samples[:, i],
        bins=25,
        stat="density",
        kde=True,
        color=palette[i],
        alpha=0.45,
        edgecolor="white",
        ax=ax
    )

    ax.axvline(
        patient_mean[i],
        color="#1f77b4",
        linewidth=2,
        label=f"mean = {patient_mean[i]:.3f}"
    )

    ax.axvline(
        patient_thresholds[i],
        color="#d62728",
        linestyle="--",
        linewidth=2,
        label=f"threshold = {patient_thresholds[i]:.3f}"
    )

    ax.set_title(
        f"{col} | true={patient_true[i]} | pred={patient_pred[i]} | std={patient_std[i]:.3f}",
        fontsize=11
    )
    ax.set_xlim(0, 1)
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Density")
    ax.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

In [17]:
import pickle

with open('model_final_mlp_drop.pkl', 'wb') as f:
    pickle.dump(model_final, f)